# Mesma coisa do experiment_9_svr_riemann_features_std, porém usando features que foram selecionadas apenas com 80% do dataset

In [6]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score

In [7]:
features_map = {
    "J. Kampe": [
        "z_term_3", "z_term_2", "d_lag_2", "d_lag_3", "d_lag_4", "z_cogram", "d_lag_16", "d_lag_5",
        "d_lag_15", "d_lag_17", "d_lag_12", "d_lag_6", "d_lag_1", "z_term_4", "z_term_6", "d_lag_18",
        "d_lag_13", "d_lag_7", "d_lag_14", "z_cogram_lag_4", "z_gram_lag_4", "z_cogram_lag_5", "z_gram_lag_5", "z_gram_lag_7",
        "z_term_5", "z_gram_lag_6", "z_cogram_lag_6", "z_cogram_lag_7", "d_lag_11", "z_cogram_lag_2", "d_lag_21", "z_cogram_lag_8",
        "z_cogram_lag_11", "z_cogram_lag_3", "d_lag_8", "z_gram_lag_2", "z_gram_lag_8", "z_gram_lag_1", "z_cogram_lag_12", "d_lag_19"
    ]
}

random_forest_features = pd.read_csv("../results/random_forest_feature_selection_v5.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection_v5.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/gevrey_method_feature_selection_v5.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/mrmr_10_features_v5.csv")["feature"].tolist()
mrmr_14_features = pd.read_csv("../results/mrmr_14_features_v5.csv")["feature"].tolist()

features_map["Random Forest (5)"] = random_forest_features[:5]
features_map["Random Forest (10)"] = random_forest_features[:10]
features_map["Random Forest Full"] = random_forest_features
features_map["Correlation"] = correlation_features
features_map["Gevrey Method"] = gevrey_method_features
features_map["Gevrey Method (10 features)"] = gevrey_method_features[:10]  # Limiting to top 10 features
features_map["mRMR (10 features)"] = mrmr_10_features

In [8]:
class DatasetService:
    OFFSET = 1000
    LIMIT  = 11_000

    def __init__(self, features: list[str]):
        self.X_df = pd.read_csv("../dataset/j_kampe.csv")
        self.y_df = pd.read_csv("../dataset/distances.csv")["distance"]
        self.features = features

    def get_train_test(self):
        X = self.X_df[self.features].to_numpy()[self.OFFSET:self.LIMIT]
        y = self.y_df.to_numpy()[self.OFFSET:self.LIMIT]

        split = int(0.8 * len(X))
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]

        return X_train, X_test, y_train, y_test

In [9]:
def run_experiment(features_map):
    results = []

    param_grid = {
        "svr__C": [0.1, 1, 10, 100],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0]
    }

    for group_name, features in features_map.items():
        print(f"Running experiment for group: {group_name} with {len(features)} features")
        data = DatasetService(features)
        X_train, X_test, y_train, y_test = data.get_train_test()

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf"))
        ])

        grid = GridSearchCV(
            pipeline,
            param_grid,
            scoring="neg_root_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            n_jobs=-1
        )

        grid.fit(X_train, y_train.ravel())

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "Group": group_name,
            "RMSE": rmse,
            "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
            "Features": len(features)
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [10]:
df_standard = run_experiment(features_map)
print(df_standard)
df_standard.to_csv("../results/experiment_9_svr_jkampe_std_80_percent.csv", index=False)

Running experiment for group: J. Kampe with 40 features
Running experiment for group: Random Forest (5) with 5 features
Running experiment for group: Random Forest (10) with 10 features
Running experiment for group: Random Forest Full with 20 features
Running experiment for group: Correlation with 44 features
Running experiment for group: Gevrey Method with 24 features
Running experiment for group: Gevrey Method (10 features) with 10 features
Running experiment for group: mRMR (10 features) with 10 features
                         Group      RMSE        R2  Best C  Best epsilon  \
0           Random Forest Full  0.030291  0.986033   100.0         0.001   
1                     J. Kampe  0.033892  0.982514    10.0         0.001   
2                  Correlation  0.072647  0.919663    10.0         0.010   
3                Gevrey Method  0.075761  0.912628    10.0         0.010   
4           mRMR (10 features)  0.109846  0.816324     1.0         0.010   
5  Gevrey Method (10 features) 